### 1 - 1

Студия Peacegaming выпустила игру World of Tractors. Участники со всего мира соревнуются, кто быстрее соберёт урожай с бесконечного поля. Что нужно сделать, чтобы игра приносила прибыль?

Что известно:

В World of Tractors играет 50 тысяч пользователей.

Каждый пользователь в среднем платит 10 долларов ежемесячно.

Маркетинговый бюджет — 200 тысяч долларов в месяц.

На поддержку серверов тратится 200 тысяч долларов (чем больше людей в игре, тем больше серверов нужно).

Зарплата программистов — 300 тысяч $.

На аренду офиса уходит в 150 тысяч $.

Запишите данные в переменные, посчитайте выручку, суммарные расходы и итоговую прибыль.

In [1]:

import pandas as pd
import numpy as np


In [2]:
n_users = 5000
revenue_per_user = 10
var_costs = pd.Series({'marketing': 200000, 'servers': 200000, })
fixed_costs = pd.Series({'salery': 300000, 'rent': 150000})

In [3]:
revenue = n_users*revenue_per_user
total_costs = var_costs.sum() + fixed_costs.sum()

In [4]:
print('Выручка:', revenue)
print('Затраты:', total_costs)
print('Итого:', revenue - total_costs)

Выручка: 50000
Затраты: 850000
Итого: -800000


### 1 - 2

Посчитайте экономику одной продажи игры World of Tractors без учёта постоянных расходов. Сохраните результат в переменной one_unit_var_costs. Выведите на экран выручку от одной продажи, переменные затраты на одну продажу и итог.

In [5]:
one_unit_var_costs = var_costs / n_users

print('Переменные затраты на одну продажу:')
print(one_unit_var_costs)

Переменные затраты на одну продажу:
marketing    40.0
servers      40.0
dtype: float64


### 1 -3

Экономика сходится. Постройте модель для расчёта объема продаж, при котором бизнес выйдет в плюс. Количество пользователей вырастет, если увеличить расходы на маркетинг.

In [6]:
def unit_economics(marketing):    
    # найдем колличество пользователей, в зависимотсти от бюджета на рекламу
    n_users = marketing / one_unit_var_costs['marketing']
    revenue = revenue_per_user * n_users
    var_costs = one_unit_var_costs * n_users
    return revenue - sum(var_costs) - sum(fixed_costs)

for m in range(300000, 1500000, 100000):
    print('Прибыль/убыток: {} при бюджете в {}'.format(unit_economics(m), m))

Прибыль/убыток: -975000.0 при бюджете в 300000
Прибыль/убыток: -1150000.0 при бюджете в 400000
Прибыль/убыток: -1325000.0 при бюджете в 500000
Прибыль/убыток: -1500000.0 при бюджете в 600000
Прибыль/убыток: -1675000.0 при бюджете в 700000
Прибыль/убыток: -1850000.0 при бюджете в 800000
Прибыль/убыток: -2025000.0 при бюджете в 900000
Прибыль/убыток: -2200000.0 при бюджете в 1000000
Прибыль/убыток: -2375000.0 при бюджете в 1100000
Прибыль/убыток: -2550000.0 при бюджете в 1200000
Прибыль/убыток: -2725000.0 при бюджете в 1300000
Прибыль/убыток: -2900000.0 при бюджете в 1400000


### 2 - 1

Вы работаете в интернет-магазине. Ваша задача проанализировать юнит-экономику и помочь маркетологам разобраться — снижать или увеличивать расходы на маркетинг. У вас есть данные о продажах и расходах. Вы также знаете, что маржинальность магазина равна 40%.

Сохраните данные о заказах из файла '/datasets/ltv_orders_2.csv' в переменную orders, информацию о расходах из файла '/datasets/ltv_costs_2.csv' — в переменную costs. Приведите даты к правильному формату, добавьте столбцы с месяцем заказа и месяцем расходов, соответственно.

Ничего выводить на экран не нужно.

In [7]:
orders = pd.read_csv('./ltv_orders_2.csv')
costs = pd.read_csv('./ltv_costs_2.csv')




In [8]:
costs.head()

,date,costs
0,2019-01-07,3085
1,2019-01-12,5594
2,2019-01-13,8523
3,2019-01-14,10356
4,2019-01-15,7455


In [9]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
costs['date'] = pd.to_datetime(costs['date'])


In [10]:
orders['order_month'] = orders['order_date'].dt.month
costs['month'] = costs['date'].dt.month



### 2 - 2

Посчитайте количество уникальных покупателей в каждой когорте.

Найдите месяц первого заказа каждого покупателя и сохраните результат в переменной first_orders. Вычислите, сколько людей совершили покупку впервые в каждом месяце. Результат сохраните в переменной cohort_sizes.

Выведите на экран первые 5 строк датафрейма из cohort_sizes.

In [11]:
first_orders = orders.groupby('uid')['order_month'].min().reset_index()
first_orders.columns = ['uid', 'first_order_month']

cohort_sizes = (
    first_orders.groupby('first_order_month')
    .agg({'uid':'nunique'})
    .reset_index()
)
cohort_sizes.columns = ['first_order_month', 'n_buyers']
print(cohort_sizes.head())

   first_order_month  n_buyers
0                  1        46
1                  2       122
2                  3       182
3                  4       173
4                  5       182


### 2 - 3

Создайте когортный отчёт, добавьте в него столбец с возрастом каждой когорты, суммарной валовой прибылью когорты в каждый месяц, посчитайте LTV.

Постройте сводную таблицу, в строках которой будет месяц когорты, в столбцах — её возраст, а в значениях — средний LTV. Все значения в таблице округлите до целого.

Напомним, маржинальность магазина равна 40%

In [12]:
margin_rate = 0.4

# добавьте месяц первого заказа в датафрейм к покупкам
orders_first_month = pd.merge(orders, first_orders, on ='uid' )


# сгруппируйте зазазы в когорты
cohorts = (
    orders_first_month.groupby(['first_order_month', 'order_month' ])
    .agg({'revenue': 'sum'})
    .reset_index()
)

In [13]:
# объедините cohort_sizes и cohorts
report = pd.merge(cohort_sizes, cohorts, on='first_order_month')

report['gp'] =  report['revenue'] * margin_rate

In [14]:
# посчитайте возраст каждой когорты в месяцах, не забудьте что возраст должен быть целым значением 

report['age'] = (report['order_month'] - report['first_order_month']) 
report['age'] = report['age'].round().astype('int')

In [15]:

# посчитайте LTV когорт
report['ltv'] = report['gp'] / report['n_buyers']

# посчитайте сводную таблицу
result = report.pivot_table(
    index='first_order_month',
    columns='age',
    values='ltv',
    aggfunc='mean',
).round()
result

age,0,1,2,3,4,5,6
first_order_month,,,,,,,
1,536.0,487.0,567.0,484.0,325.0,70.0,5.0
2,551.0,484.0,523.0,442.0,258.0,88.0,17.0
3,625.0,493.0,549.0,425.0,247.0,89.0,4.0
4,572.0,527.0,456.0,418.0,199.0,20.0,NaN
5,538.0,507.0,450.0,260.0,31.0,NaN,NaN
6,622.0,496.0,370.0,53.0,NaN,NaN,NaN
7,535.0,514.0,121.0,NaN,NaN,NaN,NaN
8,580.0,116.0,NaN,NaN,NaN,NaN,NaN
9,381.0,NaN,NaN,NaN,NaN,NaN,NaN


### 2 - 4

Посчитайте CAC и ROMI каждой когорты.

В переменной monthly_costs найдите сумму расходов на маркетинг за каждый месяц. Добавьте информацию о расходах в когортный отчёт и посчитайте CAC каждой когорты. Результат сохраните в столбце report_new['cac']. Наконец, посчитайте ROMI когорт, поделив LTV на CAC.

Выведите на экран сводную таблицу с отчётом. В строках — месяц когорты, в столбцах — возраст когорты, в значениях — накопительный средний ROMI, округлённый до двух знаков после запятой. Удалите пропущенные значения в таблице.

In [16]:
costs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    236 non-null    datetime64[ns]
 1   costs   236 non-null    int64         
 2   month   236 non-null    int32         
dtypes: datetime64[ns](1), int32(1), int64(1)
memory usage: 4.7 KB


In [18]:
monthly_costs = costs.groupby('month')['costs'].sum().reset_index()

print(monthly_costs)

   month   costs
0      1  131376
1      2  365652
2      3  574721
3      4  496602
4      5  536254
5      6  660975
6      7  706049
7      8  796449
8      9  207446


In [19]:
report_new = pd.merge(
    report, monthly_costs, left_on='first_order_month', right_on='month'
)
report_new['cac'] = report_new['costs'] / report_new['n_buyers']

report_new['romi'] = report_new['ltv'] / report_new['cac']
output = report_new.pivot_table(
    index='first_order_month', columns='age', values='romi', aggfunc='mean'
)



### 2 - 5

Посчитайте LTV средней когорты. Узнайте, сколько денег могут тратить маркетологи на привлечение одного покупателя, если хотят, чтобы реклама окупилась за 6 месяцев.

In [20]:
# посчитайте сводную таблицу с LTV
final_result = report_new.pivot_table(
    index='first_order_month',
    columns='age',
    values='ltv',
    aggfunc='mean',
)

# посчитайте накопительный LTV за 6 месяцев после первой покупки
m6_cum_ltv = final_result.cumsum(axis=1).mean(axis=0)[5]

print('Средний LTV за 6 месяцев после первой покупки:', m6_cum_ltv)

Средний LTV за 6 месяцев после первой покупки: 2358.917911699909
